# 탐지기 v2 · YOLO11s · Colab GPU 학습

**상태: 실행 준비용. 실제 v2 학습 성능은 실행 후 생성됩니다.**

1. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
2. 노트북을 실행할 저장소 체크아웃을 준비합니다. 데이터·가중치는 Drive 경로를 사용합니다.
3. 아래 셀을 순서대로 실행합니다. Drive v2 패키지 약 17GB를 `/content`에 복사하므로 시간이 걸립니다.
4. 체크포인트와 결과는 Drive의 `gitkk_archive/outputs/detector/<run_name>`에 매 epoch 저장됩니다.

저장소 기준: seed 42, 최대 50 epoch, patience 7, SGD lr 0.005/momentum 0.9/weight_decay 0.0005, cosine LR, 증강 비활성.
YOLO 조건: COCO 사전학습 `yolo11s.pt`, 640 letterbox, AutoBatch(메모리 70%), EMA 평가. optimizer parameter grouping·gradient accumulation 등은 Ultralytics 동작이며 SSD와 동일한 구현은 아닙니다.

best는 저장소 기준 **고 IoU COCO mAP@[0.75:0.95] → MAR@100** 순서로 선택합니다. Ultralytics 기본 best 기준은 사용하지 않습니다.


In [ ]:
import os, sys, subprocess
os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_CONFIG_DIR"] = "/content/yolo-settings"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics==8.4.183", "torchmetrics[detection]==1.9.0",
                       "pycocotools", "pyyaml", "pillow", "matplotlib", "pandas"])
import torch, ultralytics
assert torch.cuda.is_available(), "Colab 런타임을 GPU로 변경하세요."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "ultralytics:", ultralytics.__version__)


## 저장소 코드 준비·Drive 연결

저장소에서 직접 실행합니다. Colab에 체크아웃이 없으면 GitHub에서 가져옵니다. PR 병합 전 검토 시 SOURCE_REF를 작업 브랜치로 지정하세요.


In [ ]:
import sys, subprocess
from pathlib import Path
SOURCE_REF = "main"  # PR 검토 중에는 "37-yolo11s-training-e2e"로 지정
REPOSITORY_URL = "https://github.com/auranova1125/gitkk-pill-detection.git"
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file()), None)
if project_root is None:
    project_root = Path("/content/gitkk-pill-detection")
    if not (project_root / ".git").is_dir():
        subprocess.check_call(["git", "clone", "--branch", SOURCE_REF, REPOSITORY_URL, str(project_root)])
PROJECT_ROOT = project_root.resolve()
assert (PROJECT_ROOT / "pipeline/yolo_e2e.py").is_file(), "YOLO 코드가 있는 브랜치를 열어주세요. PR 검토 중에는 SOURCE_REF를 변경하세요."
import os
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
# 이전 프로젝트 복사본이 같은 Colab 세션에서 import된 경우 새 체크아웃 기준으로 읽습니다.
for name, module in list(sys.modules.items()):
    filename = str(getattr(module, "__file__", ""))
    if name.split(".")[0] in {"pipeline", "classifier", "detector", "data_loading", "configs", "utils"}:
        if "/content/" in filename and not filename.startswith(str(PROJECT_ROOT)):
            del sys.modules[name]
from google.colab import drive
drive.mount("/content/drive")


## 데이터 복사·무결성 검사·YOLO 라벨 변환
기존 v2 train/val을 그대로 사용하며 재분할하지 않습니다. 원본 annotation·이미지는 수정하지 않습니다. 저장소 strict loader로 그룹 중복·bbox·ID·metadata를 확인하고 YOLO class 0으로 변환합니다. bbox 이미지 크기도 전체 대조합니다.

복사가 끊겼다면 불완전한 `/content/detector-v2`를 지우고 이 셀을 다시 실행하세요. 원본 Drive 폴더는 지우지 않습니다.


In [ ]:
import json, shutil
from data_loading.yolo_detector import export_yolo_package, verify_export, V2_SPLIT_HASH
SOURCE_PACKAGE = Path("/content/drive/MyDrive/gitkk_archive/detector_data/1. selected_data/v2")
LOCAL_PACKAGE = Path("/content/detector-v2")
YOLO_PACKAGE = PROJECT_ROOT / "datasets/yolo11s-v2"
assert SOURCE_PACKAGE.is_dir(), f"Drive v2 경로를 확인하세요: {SOURCE_PACKAGE}"
if not LOCAL_PACKAGE.exists():
    required = sum(p.stat().st_size for p in SOURCE_PACKAGE.rglob("*") if p.is_file())
    free = shutil.disk_usage("/content").free
    assert free > required + 5 * 1024**3, "Colab 로컬 디스크 공간이 부족합니다."
    print(f"데이터 복사 시작: {required / 1024**3:.1f} GiB")
    shutil.copytree(SOURCE_PACKAGE, LOCAL_PACKAGE)
settings = {"package_dir": str(LOCAL_PACKAGE),
            "image_roots": {"select": str(LOCAL_PACKAGE), "codeit": str(LOCAL_PACKAGE)}}
DATA_YAML = YOLO_PACKAGE / "data.yaml"
if not YOLO_PACKAGE.exists():
    DATA_YAML = export_yolo_package(settings, YOLO_PACKAGE)
summary = verify_export(DATA_YAML)
assert summary["split_hash"] == V2_SPLIT_HASH
assert summary["num_train_images"] == 8143 and summary["num_val_images"] == 2026
print(json.dumps(summary, indent=2))


## bbox 사전 확인
원본 좌표를 그린 train/val 표본입니다. 알약 전체를 둘러싸는지 확인하고 이상하면 학습 전에 중단하세요. 이 확인은 전체 라벨의 사람 검수를 대체하지 않습니다.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
records = json.loads((YOLO_PACKAGE / "source_records.json").read_text())
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, split in enumerate(["train", "validation"]):
    samples = [r for r in records if r["split"] == split]
    for ax, record in zip(axes[row], samples[::max(1, len(samples)//3)][:3]):
        with Image.open(record["image_path"]) as image:
            ax.imshow(image.convert("RGB"))
        for x1,y1,x2,y2 in record["boxes"]:
            ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
        ax.set_title(f"{split} / id={record['image_id']}")
        ax.axis("off")
plt.tight_layout()


## 실행 설정
최초 실행은 `RESUME_RUN_NAME=None`으로 둡니다. 중단된 학습은 기존 실행 이름을 입력하면 해당 `last.pt`와 **당시 설정**을 불러옵니다. 동일 이름의 신규 학습은 허용하지 않습니다.

기본 `batch=0.7`은 GPU 메모리 70%를 목표로 자동 결정합니다. 메모리 부족 시 `batch=8` 등으로 낮춰 **새 실행**을 시작하세요. 실제 batch·누적 gradient·native optimizer 설정은 `actual_train_args.json`과 `resources.json`에 기록됩니다.


In [ ]:
import yaml
from datetime import datetime
from zoneinfo import ZoneInfo
RESUME_RUN_NAME = None  # 예: "20261103-yolo11s-det-v2-baseline-01"
OUTPUT_ROOT = Path("/content/drive/MyDrive/gitkk_archive/outputs/detector")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load((PROJECT_ROOT / "configs/detector_yolo11s.yaml").read_text())
if RESUME_RUN_NAME:
    previous = json.loads((OUTPUT_ROOT / RESUME_RUN_NAME / "resolved_config.json").read_text())
    config = {key: previous[key] for key in config}
    resume_from = OUTPUT_ROOT / RESUME_RUN_NAME / "weights/last.pt"
else:
    date = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d")
    number = 1
    while (OUTPUT_ROOT / f"{date}-yolo11s-det-v2-baseline-{number:02d}").exists():
        number += 1
    config["run_name"] = f"{date}-yolo11s-det-v2-baseline-{number:02d}"
    config["output_dir"] = str(OUTPUT_ROOT)
    config["data_yaml"] = str(DATA_YAML)
    resume_from = None
CONFIG_PATH = PROJECT_ROOT / "outputs/runtime/yolo11s-colab.yaml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(yaml.safe_dump(config, sort_keys=False))


## 학습 → best checkpoint 재평가 → 노션용 결과 생성
학습 중 매 epoch val의 box/cls/dfl loss와 저장소 지표를 계산합니다. 최종 평가는 best checkpoint를 새로 읽어 수행합니다. 독립 loss 계산에서도 모델은 eval 모드로 유지합니다.

Colab 연결이 끊겨도 Drive에 `last.pt`가 있으면 위 설정 셀에서 같은 실행 이름으로 재개할 수 있습니다. 완료 후 재평가만 실패한 경우에는 아래 복구 셀을 사용하세요.


In [ ]:
from detector.yolo_backend import train_yolo
run_dir = train_yolo(CONFIG_PATH, resume=resume_from)
print("완료:", run_dir)


## 결과 확인 및 저장
`repo/*`는 저장소의 torchmetrics COCO 평가이며 `metrics/*`는 Ultralytics 자체 평가입니다. 보고서 비교에는 동일 평가기의 `repo/*`를 사용합니다. Loss는 batch별 성분 평균의 평균으로 기록되며 SSD loss와 직접 비교하지 않습니다. val은 모델 선택에 사용되었으므로 독립 test 성능으로 표현하지 않습니다.


In [ ]:
from pathlib import Path
from google.colab import drive, files
import matplotlib.pyplot as plt
drive.mount("/content/drive")
RESULT_RUN_NAME = "20261103-yolo11s-det-v2-baseline-01"  # 새 실행 결과는 실제 실행명으로 변경
run_dir = Path("/content/drive/MyDrive/gitkk_archive/outputs/detector") / RESULT_RUN_NAME
assert (run_dir / "results.csv").is_file(), f"결과 실행명을 확인하세요: {run_dir}"
import pandas as pd
from IPython.display import display, Markdown, Image as DisplayImage
history = pd.read_csv(run_dir / "results.csv")
history.columns = history.columns.str.strip()
display(history.tail())
loss_columns = [c for c in history if c.startswith(("train/", "val/")) and "loss" in c]
history.plot(x="epoch", y=loss_columns, figsize=(12, 5), grid=True)
plt.show()
display(Markdown((run_dir / "notion_report.md").read_text()))
for image in sorted((run_dir / "best_validation").glob("val_batch*_pred.jpg"))[:2]:
    display(DisplayImage(filename=str(image)))
print("전체 산출물은 Drive에 저장되었습니다:", run_dir)
print("노션 붙여넣기/가져오기:", run_dir / "notion_report.md")
files.download(str(run_dir / "notion_report.md"))


## 선택: 학습 완료 후 최종 평가·문서 생성만 재시도
이 셀은 필요할 때만 실행하세요. 기존 결과를 덮어쓰지 않도록 새 평가 폴더를 만듭니다. 위 학습 셀은 다시 실행하지 않습니다.


In [ ]:
# 필요한 경우 아래 코드를 주석 해제하세요.
# from detector.yolo_backend import evaluate_yolo_checkpoint, write_report
# run_dir = OUTPUT_ROOT / "여기에-완료된-실행명"
# native = json.loads((run_dir / "actual_train_args.json").read_text())
# evaluation_dir = run_dir / ("best_validation_retry_" + datetime.now().strftime("%H%M%S"))
# values = evaluate_yolo_checkpoint(run_dir / "weights/best.pt", DATA_YAML,
#     device="0", batch=int(native["batch"]), imgsz=int(native["imgsz"]), destination=evaluation_dir)
# write_report(run_dir, values)


## 후속 분석 체크리스트
- `results.png`와 loss 곡선에서 과적합·학습 불안정을 확인합니다.
- 검증 예측 그림에서 미탐·중복탐지·정밀 위치 오차를 모읍니다.
- 동일 v2 split의 SSD 결과와 고 IoU mAP, recall, 객체 수 오차를 비교합니다. 입력 크기와 COCO 사전학습 차이를 함께 적습니다.
- 동일 GPU에서 추론 시간·peak memory를 비교합니다. `resources.json`과 `best_validation/evaluation.json`에 측정 조건이 있습니다.
- YOLO checkpoint는 기존 torchvision 통합 loader와 형식이 다릅니다. 분류기 연결·end-to-end 평가는 별도 후속 범위입니다.

공식 참고: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [학습 설정](https://docs.ultralytics.com/modes/train/).
